In [33]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


## Step 1 — TabZilla benchmark datasets

This step defines the OpenML datasets included in the TabZilla GAM benchmark. The dataset identifiers are fixed in advance so that the same collection of binary classification tasks can be evaluated across training objectives and model classes.

Datasets are subsequently retrieved directly from OpenML. Eligibility checks, including binary-outcome verification and minimum sample-size requirements, are applied during benchmark execution.

In [34]:
OPENML_IDS = [
    1120,   # Magic telescope
    1053,   # openml__jm1__3904
    4532,   # higgs
    4534,   # phishing websites
    1489,   # phoneme
    1502,   # skin segmentation
    1590,   # adult income
    45072,  # airlines
    151,    # electricity
    4135,   # Amazon_employee_access
    40978,  # internet advertisements
    41434,  # click prediction small
    41150,  # miniBooNE
    40536,  # speeddating
    1043,   # ada agnostic
    1462,   # banknote authentication
    41142,  # christine
    40701,  # churn
    31,     # credit-g
    1471,   # eeg-state
    846,    # elevators
    1038,   # gina agnostic
    821,    # house 16H
    41143,  # jasmine
    1067,   # kc1
    1485,   # madelon
    24,     # mushroom
    1116,   # musk
    1486,   # nomao
    23517,  # numerai28.6
    1487,   # ozone
    1068,   # pc1
    1050,   # pc3
    1049,   # pc4
    41145,  # philippine
    871,    # pollen
    312,    # scene
    38,     # sick
    44,     # spambase
    1570,   # wilt
    45035,  # Albert
    1461,   # bank marketing
    1036,   # sylvia agnostic
    41146,  # sylvine
]

## Step 2 — Cross-validation splits

This step defines the outer train/test partitions used throughout the benchmark. Each dataset is split using stratified 5-fold cross-validation with a fixed random seed.

For each of the five runs, one fold is used as the independent test set and the remaining four folds form the training set. This rotating-fold design ensures that every observation is used for testing exactly once.

The same fold construction is used across training objectives and can be shared across model classes, allowing comparisons to be performed on identical participant-level or observation-level test sets.

In [35]:

from sklearn.model_selection import StratifiedKFold
import numpy as np

def make_5fold_indices(y: np.ndarray, seed: int):
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )
    return [test_idx for _, test_idx in skf.split(np.zeros_like(y), y)]


def indices_for_run_train_test(folds, run_id: int):
    """
    5-fold rotation:
    - run_id fold = test
    - all other folds = train
    """
    test_fold = run_id % 5

    test_inds = folds[test_fold]

    train_inds = np.concatenate(
        [folds[i] for i in range(5) if i != test_fold],
        axis=0,
    )

    return train_inds, test_inds

## Step 3 — Shared preprocessing

This step defines the preprocessing pipeline applied to each OpenML dataset before XGBoost training.

Feature types are inferred from the outer training fold. Binary variables are retained as single features, string variables and numeric variables with at most 20 unique values are treated as categorical, and remaining numeric variables are treated as continuous.

Categorical variables are restricted to their 10 most frequent levels, with less frequent levels grouped into an `OTHER` category, and are subsequently one-hot encoded. Continuous variables are median-imputed and standardized. Binary variables are retained without scaling.

All preprocessing transformations are fitted on the outer training fold only and are then applied unchanged to the corresponding test fold. The resulting matrices are converted to dense `float32` arrays for XGBoost training and evaluation.

In [36]:

import numpy as np
import pandas as pd

from dataclasses import dataclass
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import TensorDataset, DataLoader


def _to_dense_float32(X):
    if sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)


def _make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse=False)


def infer_feature_types(X: pd.DataFrame, *, numeric_cat_max_unique: int = 20):
    binary_cols, categorical_cols, numeric_cols = [], [], []

    for c in X.columns:
        s = X[c]
        nunq_including_nan = pd.Series(s).nunique(dropna=False)

        if nunq_including_nan == 2:
            binary_cols.append(c)
            continue

        dtype_name = str(s.dtype)

        if dtype_name in ("object", "category", "string"):
            categorical_cols.append(c)
            continue

        if pd.api.types.is_numeric_dtype(s):
            if pd.Series(s).nunique(dropna=False) <= numeric_cat_max_unique:
                categorical_cols.append(c)
            else:
                numeric_cols.append(c)
            continue

        categorical_cols.append(c)

    return binary_cols, categorical_cols, numeric_cols


class TopKCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, top_k: int = 10):
        self.top_k = int(top_k)
        self.keep_values_ = None
        self.columns_ = None

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)
        self.keep_values_ = {}

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_values_[c] = set(vc.head(self.top_k).index.tolist())

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = pd.DataFrame(index=X_df.index)

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            keep = self.keep_values_[c]
            out[c] = s.where(s.isin(keep), "__OTHER__")

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


class BinaryPassthroughEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.columns_ = None
        self.fill_values_ = {}
        self.value_maps_ = {}

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)

        for c in self.columns_:
            s = X_df[c]
            non_missing = s[~pd.isna(s)]

            if len(non_missing) == 0:
                self.fill_values_[c] = 0
                self.value_maps_[c] = {}
                continue

            mode_vals = non_missing.mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else non_missing.iloc[0]
            self.fill_values_[c] = fill_val

            seen = []
            for v in non_missing:
                if v not in seen:
                    seen.append(v)

            if len(seen) == 1:
                mapping = {seen[0]: 0.0}
            else:
                mapping = {seen[0]: 0.0, seen[1]: 1.0}

            self.value_maps_[c] = mapping

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = np.zeros((len(X_df), len(self.columns_)), dtype=np.float32)

        for j, c in enumerate(self.columns_):
            s = X_df[c].copy()
            fill_val = self.fill_values_[c]
            mapping = self.value_maps_[c]

            s = s.where(~pd.isna(s), fill_val)
            default_code = mapping.get(fill_val, 0.0)

            out[:, j] = s.map(lambda v: mapping.get(v, default_code)).astype(np.float32).to_numpy()

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


@dataclass
class PreprocessorBundle:
    binary_cols: list
    categorical_cols: list
    numeric_cols: list
    preprocessor_base: ColumnTransformer
    feature_names: list


def _fit_shared_preprocessor_bundle(
    X_tr: pd.DataFrame,
    *,
    numeric_cat_max_unique: int = 20,
    top_k_categories: int = 10,
):

    binary_cols, categorical_cols, numeric_cols = infer_feature_types(
        X_tr, numeric_cat_max_unique=numeric_cat_max_unique
    )

    cat_pipe = Pipeline(
        steps=[
            ("topk", TopKCategoryGrouper(top_k=top_k_categories)),
            ("ohe", _make_ohe()),
        ]
    )

    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    bin_pipe = Pipeline(
        steps=[
            ("binary", BinaryPassthroughEncoder()),
        ]
    )

    preprocessor_base = ColumnTransformer(
        transformers=[
            ("cat", cat_pipe, categorical_cols),
            ("num", num_pipe, numeric_cols),
            ("bin", bin_pipe, binary_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

    preprocessor_base.fit(X_tr)

    try:
        feature_names = list(preprocessor_base.get_feature_names_out())
    except Exception:
        feature_names = [f"x{i}" for i in range(preprocessor_base.transform(X_tr.iloc[:1]).shape[1])]

    return PreprocessorBundle(
        binary_cols=binary_cols,
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        preprocessor_base=preprocessor_base,
        feature_names=feature_names,
    )


def fit_train_preprocessor_and_transform(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
):
    bundle = _fit_shared_preprocessor_bundle(X_tr)

    X_tr_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_tr))
    X_te_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_te))

    if np.isnan(X_tr_enc).any() or np.isnan(X_te_enc).any():
        raise RuntimeError("Preprocessing produced NaNs.")

    return bundle, X_tr_enc, X_te_enc

## Step 4 — Global configuration

This step defines the benchmark-wide settings used throughout the XGBoost experiments.

It specifies the computational device, random seeds, minimum dataset size, decision-threshold band, threshold-grid resolution, and local calibration settings.

For the BCE-trained XGBoost baseline, the maximum number of boosting rounds and early-stopping criterion are defined. For Smooth Net Benefit continuation, the maximum number of additional trees, evaluation frequency, and early-stopping settings are specified.

Smooth Net Benefit training uses an inverse-temperature annealing schedule of 1, 4, and 10. Three Hessian variants are evaluated during decision-focused XGBoost training: `absw`, `fixed025`, and `true`.

In [37]:
import random

import numpy as np
import torch
import xgboost as xgb

from nbloss.trainer import set_seed


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

# ---- Global benchmark settings
SEED_GLOBAL = 1234
MODEL_SEED = 4242
MIN_ROWS = 1000
BATCH = 1024

set_seed(SEED_GLOBAL)
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---- Decision-band settings
BAND_HALF_WIDTH = 0.025
MID_PREV_LOW, MID_PREV_HIGH = 0.40, 0.60

TRAIN_RANGE_POINTS = 11
VAL_RANGE_POINTS = 5
TEST_RANGE_POINTS = 201

# ---- Local calibration settings
LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500

# ---- XGB BCE baseline control
BCE_NUM_BOOST_ROUND = 1000
BCE_EARLY_STOPPING_ROUNDS = 50

# ---- XGB NB stage control
NB_ADDITIONAL_TREES_CAP = 600
NB_EARLY_STOP_TREES = 50
NB_EVAL_EVERY_TREES = 10

# ---- Annealed NB inverse temperatures
INVERSE_TEMPS = (1.0, 4.0, 10.0)
NB_ANNEAL_INV_TEMPS = INVERSE_TEMPS

# ---- XGB Hessian variants
HESSIAN_MODES = ["absw", "fixed025", "true"]

def band_from_t_ref(t_ref: float, half_width: float = BAND_HALF_WIDTH) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)

## Step 5 — OpenML loading, output utilities, and threshold specification

This step defines utility functions for loading benchmark datasets from OpenML and managing benchmark outputs.

For each dataset, the predictors and binary target are retrieved together with metadata such as sample size, number of raw predictors, prevalence, and class labels.

Reference decision thresholds are derived from the prevalence of the outer training fold. The prevalence threshold is evaluated for every dataset. For datasets with prevalence outside the central 40%–60% range, an additional inverse-prevalence threshold is evaluated.

For each reference threshold, Net Benefit is evaluated across a decision-relevant interval of ±0.025 around the reference threshold.

Helper functions for reproducible data loading and incremental result storage are also defined in this step.

In [38]:
from pathlib import Path

import numpy as np
import pandas as pd
import openml
import torch

from torch.utils.data import TensorDataset, DataLoader


# ============================================================================
# Output helpers
# ============================================================================

def _append_csv_safely(df: pd.DataFrame, path: Path):
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def _read_done_registry(path_done: Path) -> set[int]:
    if not path_done.exists():
        return set()

    try:
        df = pd.read_csv(path_done)
        return set(df["openml_id"].astype(int).tolist())
    except Exception:
        return set()


def _mark_dataset_done(openml_id: int, name: str, path_done: Path):
    _append_csv_safely(
        pd.DataFrame([{"openml_id": int(openml_id), "name": str(name)}]),
        path_done,
    )


# ============================================================================
# OpenML loader
# ============================================================================

def load_openml_dataset(did: int):
    ds = openml.datasets.get_dataset(did)

    X_df, y_raw, _, _ = ds.get_data(
        target=ds.default_target_attribute,
        dataset_format="dataframe",
        include_row_id=False,
        include_ignore_attribute=False,
    )

    y_series = pd.Series(y_raw)

    if (
        y_series.dtype.kind in ("U", "S", "O", "b")
        or str(y_series.dtype).startswith("category")
    ):
        y_cat = y_series.astype("category")
        classes = list(y_cat.cat.categories)
        y = y_cat.cat.codes.to_numpy(dtype=np.float32)
    else:
        y = y_series.astype("int64").to_numpy(dtype=np.float32)
        classes = ["0", "1"]

    prev = float((y == 1).mean())

    meta = dict(
        openml_id=int(ds.dataset_id),
        name=str(ds.name),
        n_rows=int(len(y)),
        n_features=int(X_df.shape[1]),
        prevalence=prev,
        pos_label=(classes[1] if len(classes) == 2 else "1"),
        neg_label=(classes[0] if len(classes) == 2 else "0"),
    )

    return ds, X_df, y, meta


# ============================================================================
# PyTorch loader helper
# ============================================================================

def make_loader(X, y, batch=BATCH, shuffle=False, seed=SEED_GLOBAL):
    g = torch.Generator().manual_seed(seed)

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        generator=g,
    )


# ============================================================================
# Threshold helpers
# ============================================================================

def threshold_specs_from_prevalence(
    prev: float,
    *,
    mid_prev_low: float = MID_PREV_LOW,
    mid_prev_high: float = MID_PREV_HIGH,
):
    specs = [
        {
            "threshold_name": "prev",
            "threshold": float(prev),
        }
    ]

    if prev < mid_prev_low or prev > mid_prev_high:
        specs.append(
            {
                "threshold_name": "inverse",
                "threshold": float(1.0 - prev),
            }
        )

    return specs

## Step 6 — BCE XGBoost baseline selection and refit

This step defines the baseline XGBoost training procedure using binary logloss.

A predefined grid of XGBoost hyperparameters is evaluated on validation data. Candidate configurations vary tree depth, learning rate, minimum child weight, L2 regularization, and minimum split-loss reduction.

For each candidate configuration, boosting is performed with early stopping based on validation logloss. The configuration with the lowest validation logloss is selected.

The selected hyperparameters and number of boosting rounds are subsequently used to refit the BCE baseline model on the combined development data. This model provides both the baseline prediction model and the warm start for subsequent Smooth Net Benefit training.

In [39]:


import numpy as np
import xgboost as xgb


def _normalize_xgb_params(params: dict) -> dict:
    p = dict(params)

    if "learning_rate" in p:
        p["eta"] = float(p["learning_rate"])

    if "eta" in p and "learning_rate" not in p:
        p["learning_rate"] = float(p["eta"])

    return p


def baseline_hyperparameter_grid():
    grid = []

    for max_depth in [3, 4]:
        for learning_rate in [0.05, 0.10]:
            for min_child_weight in [1.0, 5.0]:
                for reg_lambda in [1.0, 5.0]:
                    for gamma in [0.0, 1.0]:
                        for subsample in [1.0]:
                            grid.append(
                                {
                                    "max_depth": int(max_depth),
                                    "learning_rate": float(learning_rate),
                                    "min_child_weight": float(min_child_weight),
                                    "subsample": float(subsample),
                                    "colsample_bytree": 1.0,
                                    "reg_lambda": float(reg_lambda),
                                    "reg_alpha": 0.0,
                                    "gamma": float(gamma),
                                    "tree_method": "hist",
                                }
                            )

    return grid


def xgb_fit_baseline_with_es(
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    *,
    params: dict,
    seed: int,
    num_boost_round_cap: int = BCE_NUM_BOOST_ROUND,
    early_stopping_rounds: int = BCE_EARLY_STOPPING_ROUNDS,
):
    p = _normalize_xgb_params(params)
    p = dict(p)

    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(X_train_enc, label=y_train)
    dvalid = xgb.DMatrix(X_valid_enc, label=y_valid)

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(num_boost_round_cap),
        evals=[(dvalid, "valid")],
        verbose_eval=False,
        early_stopping_rounds=int(early_stopping_rounds),
    )

    best_iter = getattr(booster, "best_iteration", None)
    best_score = getattr(booster, "best_score", None)

    if best_iter is None:
        best_iter = int(num_boost_round_cap) - 1

    if best_score is None:
        preds = np.clip(booster.predict(dvalid), 1e-12, 1.0 - 1e-12)
        best_score = float(
            -np.mean(
                y_valid * np.log(preds)
                + (1.0 - y_valid) * np.log(1.0 - preds)
            )
        )

    best_trees = int(best_iter) + 1
    best_logloss = float(best_score)

    return booster, best_trees, best_logloss


def select_baseline_by_val_logloss_with_broadcast(
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    *,
    seed: int,
):
    grid = baseline_hyperparameter_grid()
    best = None

    print(
        f"\n[Inner selection][Baseline XGB] "
        f"Evaluating {len(grid)} configs (metric=VAL logloss) ..."
    )

    rows = []

    for i, cfg in enumerate(grid, start=1):
        booster, best_trees, val_logloss = xgb_fit_baseline_with_es(
            X_train_enc,
            y_train,
            X_valid_enc,
            y_valid,
            params=cfg,
            seed=seed,
        )

        rows.append(
            {
                "grid_i": int(i),
                "val_logloss": float(val_logloss),
                "best_trees": int(best_trees),
                "params": dict(cfg),
            }
        )

        print(
            f"[Baseline {i:03d}/{len(grid)}] "
            f"val_logloss={val_logloss:.6f} | best_trees={best_trees:4d} | "
            f"max_depth={cfg['max_depth']} | lr={cfg['learning_rate']:.2g} | "
            f"min_child_weight={cfg['min_child_weight']} | "
            f"reg_lambda={cfg['reg_lambda']} | gamma={cfg['gamma']} | "
            f"subsample={cfg['subsample']} | colsample={cfg['colsample_bytree']}"
        )

        if (best is None) or (val_logloss < best[0]):
            best = (float(val_logloss), dict(cfg), int(best_trees))

    best_val_logloss, best_params, best_trees = best

    print(
        f"\n[Baseline BEST] val_logloss={best_val_logloss:.6f} | "
        f"best_trees={best_trees} | params={best_params}"
    )

    return best_params, best_trees, best_val_logloss, rows


def refit_baseline_on_trainval(
    X_trainval_enc,
    y_trainval,
    *,
    params: dict,
    n_trees: int,
    seed: int,
):
    p = _normalize_xgb_params(params)
    p = dict(p)

    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrainval = xgb.DMatrix(X_trainval_enc, label=y_trainval)

    booster = xgb.train(
        params=p,
        dtrain=dtrainval,
        num_boost_round=int(n_trees),
        verbose_eval=False,
    )

    return booster

## Step 7 — Smooth Net Benefit XGBoost continuation

This step defines the decision-focused XGBoost training procedure.

Smooth Net Benefit training starts from the BCE-trained XGBoost model and adds additional trees using a differentiable approximation to Net Benefit over the decision-relevant threshold band.

Training proceeds through an inverse-temperature annealing schedule of 1, 4, and 10. Increasing the inverse temperature makes the Smooth Net Benefit objective progressively closer to the original threshold-based Net Benefit rule.

At each annealing stage, model checkpoints are evaluated using Net Benefit on validation data. Early stopping is based on the absence of improvement in validation Net Benefit.

Three alternatives for the second-order term of the custom XGBoost objective are supported: the absolute Hessian (`absw`), a fixed Hessian of 0.25 (`fixed025`), and the exact second derivative (`true`).

In [40]:

import numpy as np
import pandas as pd
import xgboost as xgb

from nbloss.xgboost_objectives import (
    make_xgb_smooth_net_benefit_range_objective,
    net_benefit_hard_band_xgb,
)


NB_ANNEAL_INV_TEMPS = tuple(INVERSE_TEMPS)

NB_PATIENCE_BY_INV_TEMP = {
    1.0: 40,
    4.0: 20,
    10.0: 20,
}


def _inv_temp_suffix(inv_temp: float) -> str:
    if np.isclose(inv_temp, 1.0):
        return "it1"
    if np.isclose(inv_temp, 4.0):
        return "it4"
    if np.isclose(inv_temp, 10.0):
        return "it10"

    return f"it{str(inv_temp).replace('.', '')}"


def nb_hyperparameter_grid_annealed():
    grid = []

    for max_depth in [3, 4]:
        for learning_rate in [0.05, 0.10]:
            for min_child_weight in [1.0, 5.0]:
                for reg_lambda in [1.0, 5.0]:
                    for gamma in [0.0, 1.0]:
                        for subsample in [1.0]:
                            cfg = {
                                "anneal_inv_temps": tuple(NB_ANNEAL_INV_TEMPS),
                                "max_depth": int(max_depth),
                                "learning_rate": float(learning_rate),
                                "min_child_weight": float(min_child_weight),
                                "reg_lambda": float(reg_lambda),
                                "reg_alpha": 0.0,
                                "gamma": float(gamma),
                                "subsample": float(subsample),
                                "colsample_bytree": 1.0,
                                "additional_trees_cap": int(NB_ADDITIONAL_TREES_CAP),
                                "eval_every_trees": int(NB_EVAL_EVERY_TREES),
                            }

                            for inv_temp, patience in NB_PATIENCE_BY_INV_TEMP.items():
                                cfg[f"patience_{_inv_temp_suffix(inv_temp)}"] = int(patience)

                            grid.append(cfg)

    return grid


def nb_hyperparameter_grid_fixed_temp():
    return nb_hyperparameter_grid_annealed()


def warm_start_baseline_train_only(
    X_train_enc,
    y_train,
    *,
    baseline_params: dict,
    baseline_trees: int,
    seed: int = MODEL_SEED,
):
    p = _normalize_xgb_params(dict(baseline_params))

    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(baseline_trees),
        verbose_eval=False,
    )

    return booster


def _make_nb_stage_params_from_baseline(
    baseline_params: dict,
    nb_cfg: dict,
    *,
    seed: int,
):
    p = dict(baseline_params)

    p["max_depth"] = int(nb_cfg["max_depth"])
    p["learning_rate"] = float(nb_cfg["learning_rate"])
    p["min_child_weight"] = float(nb_cfg["min_child_weight"])
    p["reg_lambda"] = float(nb_cfg["reg_lambda"])
    p["reg_alpha"] = float(nb_cfg["reg_alpha"])
    p["gamma"] = float(nb_cfg["gamma"])
    p["subsample"] = float(nb_cfg["subsample"])
    p["colsample_bytree"] = float(nb_cfg["colsample_bytree"])

    p["objective"] = "binary:logistic"
    p["tree_method"] = "hist"
    p["seed"] = int(seed)
    p["random_state"] = int(seed)

    return _normalize_xgb_params(p)


def _patience_for_inv_temp(inv_temp: float, cfg: dict) -> int:
    suffix = _inv_temp_suffix(inv_temp)
    return int(cfg.get(f"patience_{suffix}", NB_PATIENCE_BY_INV_TEMP.get(float(inv_temp), 20)))


def nb_continue_annealed_with_stage_commit(
    *,
    booster0,
    X_train_enc,
    y_train,
    X_valid_enc,
    y_valid,
    t_min,
    t_max,
    anneal_inv_temps,
    num_points_train,
    n_grid_eval,
    additional_trees_cap,
    eval_every_trees,
    params_nb_stage,
    hessian_mode,
    cfg,
):
    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    dvalid = xgb.DMatrix(
        X_valid_enc,
        label=np.asarray(y_valid, dtype=np.float32),
    )

    eval_every_trees = int(max(1, eval_every_trees))
    additional_trees_cap = int(additional_trees_cap)

    margins0 = booster0.predict(
        dvalid,
        output_margin=True,
    )

    global_best_nb = net_benefit_hard_band_xgb(
        margins0,
        y_valid,
        t_min=float(t_min),
        t_max=float(t_max),
        n_grid=int(n_grid_eval),
        tau_eval=1.0,
        input_is_margin=True,
    )

    global_best_booster = booster0
    total_added_trained = 0
    total_committed_added = 0
    stage_info = []

    for inv_temp in anneal_inv_temps:
        inv_temp = float(inv_temp)

        suffix = _inv_temp_suffix(inv_temp)
        patience_checks = _patience_for_inv_temp(inv_temp, cfg)

        objective = make_xgb_smooth_net_benefit_range_objective(
            t_min=float(t_min),
            t_max=float(t_max),
            num_points=int(num_points_train),
            inv_temp=float(inv_temp),
            hessian_mode=str(hessian_mode),
        )

        stage_booster = global_best_booster
        stage_best_booster = global_best_booster
        stage_best_nb = float(global_best_nb)

        stage_added_trained = 0
        stage_best_added = 0
        no_improve = 0

        while stage_added_trained < additional_trees_cap:
            step = min(
                eval_every_trees,
                additional_trees_cap - stage_added_trained,
            )

            stage_booster = xgb.train(
                params=params_nb_stage,
                dtrain=dtrain,
                num_boost_round=int(step),
                obj=objective,
                xgb_model=stage_booster,
                verbose_eval=False,
            )

            stage_added_trained += int(step)
            total_added_trained += int(step)

            margins = stage_booster.predict(
                dvalid,
                output_margin=True,
            )

            val_nb = net_benefit_hard_band_xgb(
                margins,
                y_valid,
                t_min=float(t_min),
                t_max=float(t_max),
                n_grid=int(n_grid_eval),
                tau_eval=1.0,
                input_is_margin=True,
            )

            if val_nb > stage_best_nb + 1e-10:
                stage_best_nb = float(val_nb)
                stage_best_booster = stage_booster
                stage_best_added = int(stage_added_trained)
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience_checks:
                break

        improved_global = stage_best_nb > global_best_nb + 1e-10

        if improved_global:
            global_best_nb = float(stage_best_nb)
            global_best_booster = stage_best_booster
            committed_added_this_stage = int(stage_best_added)
        else:
            committed_added_this_stage = 0

        total_committed_added += int(committed_added_this_stage)

        stage_info.append(
            {
                "inv_temp": float(inv_temp),
                "suffix": suffix,
                "patience_checks": int(patience_checks),
                "stage_added_trained": int(stage_added_trained),
                "stage_best_added": int(stage_best_added),
                "committed_added": int(committed_added_this_stage),
                "stage_best_val_nb": float(stage_best_nb),
                "global_best_val_nb_after_stage": float(global_best_nb),
                "improved_global": bool(improved_global),
            }
        )

        print(
            f"    [anneal inv_temp={inv_temp:g}] "
            f"stage_best_nb={stage_best_nb:+.6f} | "
            f"stage_best_added={stage_best_added:4d} | "
            f"committed_added={committed_added_this_stage:4d} | "
            f"committed={improved_global} | "
            f"global_best_nb={global_best_nb:+.6f}"
        )

    return (
        global_best_booster,
        int(total_committed_added),
        float(global_best_nb),
        stage_info,
    )

## Step 8 — Local calibration for XGBoost

This step defines two post-hoc local calibration procedures for the BCE-trained XGBoost model: local temperature scaling and local Platt scaling.

For each reference decision threshold, a subset of the outer training data is selected around the corresponding predicted-risk region. The local interval is expanded when necessary until sufficient positive and negative observations are available.

Temperature scaling estimates a single multiplicative temperature parameter for the XGBoost logits. Platt scaling estimates both a slope and an intercept.

The fitted calibration transformation is then applied to predictions on the independent outer test fold. Decision performance is evaluated using Net Benefit over the same threshold band used for the other model variants.

In [41]:

import numpy as np
import torch
import torch.nn as nn

from nbloss.xgboost_objectives import net_benefit_hard_band_xgb


def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1)
    )

    return (
        torch.sigmoid(logits_t)
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]

    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))

    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)

        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (
            n_pos >= int(min_pos)
            and n_neg >= int(min_neg)
        )

        used_full_range = (
            low <= 0.0
            and high >= 1.0
        )

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(
            bce(logits, y)
            .detach()
            .cpu()
            .item()
        )

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite temperature loss: {loss.detach().item()}"
            )

        loss.backward()

        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(
            min=1e-6,
            max=1e6,
        )

        nll_after = float(
            bce(logits / temperature, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = (
        np.asarray(logits_np, dtype=np.float64)
        .reshape(-1)
        .copy()
    )

    probs_reference_np = np.asarray(
        probs_reference_np,
        dtype=float,
    ).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(
            bce(logits, y)
            .detach()
            .cpu()
            .item()
        )

    slope = torch.ones(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    intercept = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)

        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite Platt loss: {loss.detach().item()}"
            )

        loss.backward()

        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept

        nll_after = float(
            bce(calibrated_logits, y)
            .detach()
            .cpu()
            .item()
        )

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = (
        np.asarray(logits_np, dtype=np.float64)
        .reshape(-1)
        .copy()
    )

    probs_reference_np = np.asarray(
        probs_reference_np,
        dtype=float,
    ).reshape(-1)

    mask = (
        (probs_reference_np >= float(low))
        & (probs_reference_np <= float(high))
    )

    logits_out[mask] = (
        float(slope) * logits_out[mask]
        + float(intercept)
    )

    return logits_out, mask


def evaluate_nb_from_xgb_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    return net_benefit_hard_band_xgb(
        logits_np,
        y_np,
        t_min=float(thresh_min),
        t_max=float(thresh_max),
        n_grid=int(num_points),
        tau_eval=1.0,
        input_is_margin=True,
    )


def evaluate_bce_xgb_local_calibration(
    *,
    train_logits_base: np.ndarray,
    y_train: np.ndarray,
    test_logits_base: np.ndarray,
    y_test: np.ndarray,
    t_ref: float,
    t_min: float,
    t_max: float,
) -> dict:
    """
    Fit local temperature scaling and local Platt scaling on BCE-XGB train logits,
    then evaluate calibrated BCE-XGB logits on the test set.

    The local calibration range is selected using BCE-XGB train probabilities.
    The same probability range is applied to BCE-XGB test probabilities.

    No NB-XGB logits are used or modified here.
    """
    train_logits_base = np.asarray(
        train_logits_base,
        dtype=np.float64,
    ).reshape(-1)

    test_logits_base = np.asarray(
        test_logits_base,
        dtype=np.float64,
    ).reshape(-1)

    y_train = np.asarray(y_train).reshape(-1)
    y_test = np.asarray(y_test).reshape(-1)

    train_probs_base = sigmoid_np(train_logits_base)
    test_probs_base = sigmoid_np(test_logits_base)

    local_range = find_local_calibration_range(
        train_probs_base,
        y_train,
        t_ref=float(t_ref),
    )

    local_mask_train = np.asarray(
        local_range["mask"],
        dtype=bool,
    )

    if local_mask_train.sum() == 0:
        raise RuntimeError(
            "Local calibration mask is empty on the BCE-XGB training set."
        )

    # --------------------------------------------------
    # Uncalibrated BCE-XGB NB
    # --------------------------------------------------
    nb_uncalibrated = evaluate_nb_from_xgb_logits_np(
        test_logits_base,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    # --------------------------------------------------
    # Local temperature scaling
    # --------------------------------------------------
    temp_fit = fit_temperature_from_logits_np(
        train_logits_base[local_mask_train],
        y_train[local_mask_train],
    )

    test_logits_temp, test_mask_temp = apply_local_temperature_to_logits(
        test_logits_base,
        test_probs_base,
        low=float(local_range["low"]),
        high=float(local_range["high"]),
        temperature=float(temp_fit["temperature"]),
    )

    nb_temp_scaled = evaluate_nb_from_xgb_logits_np(
        test_logits_temp,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    # --------------------------------------------------
    # Local Platt scaling
    # --------------------------------------------------
    platt_fit = fit_platt_from_logits_np(
        train_logits_base[local_mask_train],
        y_train[local_mask_train],
    )

    test_logits_platt, test_mask_platt = apply_local_platt_to_logits(
        test_logits_base,
        test_probs_base,
        low=float(local_range["low"]),
        high=float(local_range["high"]),
        slope=float(platt_fit["platt_slope"]),
        intercept=float(platt_fit["platt_intercept"]),
    )

    nb_platt_scaled = evaluate_nb_from_xgb_logits_np(
        test_logits_platt,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    return {
        "local_calib_low": float(local_range["low"]),
        "local_calib_high": float(local_range["high"]),
        "local_calib_half_width": float(local_range["half_width"]),
        "local_calib_n": int(local_range["n"]),
        "local_calib_n_pos": int(local_range["n_pos"]),
        "local_calib_n_neg": int(local_range["n_neg"]),
        "local_calib_met_minimum": bool(local_range["met_minimum"]),
        "local_calib_used_full_range": bool(local_range["used_full_range"]),

        "bce_nb_uncalibrated": float(nb_uncalibrated),

        "temp_temperature": float(temp_fit["temperature"]),
        "temp_nll_before": float(temp_fit["nll_before"]),
        "temp_nll_after": float(temp_fit["nll_after"]),
        "temp_test_mask_n": int(np.sum(test_mask_temp)),
        "bce_nb_temp_scaled": float(nb_temp_scaled),

        "platt_slope": float(platt_fit["platt_slope"]),
        "platt_intercept": float(platt_fit["platt_intercept"]),
        "platt_nll_before": float(platt_fit["nll_before"]),
        "platt_nll_after": float(platt_fit["nll_after"]),
        "platt_test_mask_n": int(np.sum(test_mask_platt)),
        "bce_nb_platt_scaled": float(nb_platt_scaled),
    }

## Step 9 — Inner cross-validation for BCE and Smooth Net Benefit XGBoost

This step defines the nested model-selection procedure used within each outer training fold.

First, stratified inner cross-validation is used to select the BCE-XGBoost hyperparameters by mean validation logloss. The median number of baseline boosting rounds across the inner folds is retained for subsequent refitting.

For each Hessian variant, Smooth Net Benefit continuation hyperparameters are then evaluated using the same inner folds. Candidate configurations are compared using validation Net Benefit over the target decision band.

After the baseline and Smooth Net Benefit hyperparameters have been selected, the BCE model is refitted on the complete outer training fold and the selected Smooth Net Benefit continuation is applied from that fitted baseline.

The independent outer test fold is not used for either hyperparameter selection or model selection.

In [44]:

from sklearn.model_selection import StratifiedKFold

import numpy as np
import pandas as pd
import xgboost as xgb

from nbloss.xgboost_objectives import (
    make_xgb_smooth_net_benefit_range_objective,
    net_benefit_hard_band_xgb,
)


def make_inner_cv_folds(
    y_train: np.ndarray,
    *,
    seed: int,
    n_splits: int = 5,
):
    y_train = np.asarray(y_train).astype(int)

    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    return [
        (tr, va)
        for tr, va in skf.split(np.zeros_like(y_train), y_train)
    ]


def select_baseline_and_nb_inner_cv(
    X_train_enc: np.ndarray,
    y_train: np.ndarray,
    *,
    t_min: float,
    t_max: float,
    hessian_mode: str,
    inner_cv_splits: int = 5,
    seed: int = SEED_GLOBAL,
):
    """
    Select baseline XGBoost and annealed NB-XGBoost by inner CV.

    Returns
    -------
    best_baseline_params:
        Hyperparameters selected by validation logloss.

    median_baseline_trees:
        Median number of BCE baseline trees across inner folds.

    best_nb_cfg:
        NB-stage hyperparameters selected by validation hard NB.
        Also includes:
        - _baseline_params_for_nb_stage
        - median_added_trees
        - median_added_trees_it1
        - median_added_trees_it4
        - median_added_trees_it10

    median_added_trees:
        Median total number of committed NB trees across inner folds.
    """
    X_train_enc = np.asarray(X_train_enc, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)

    folds = make_inner_cv_folds(
        y_train,
        seed=int(seed) + 111,
        n_splits=int(inner_cv_splits),
    )

    # =========================================================
    # 1) Baseline selection by validation logloss
    # =========================================================

    base_grid = baseline_hyperparameter_grid()
    best_base = None
    best_base_fold_trees = None

    print(
        f"\n[Inner CV][Baseline] {len(base_grid)} configs | "
        f"metric=VAL logloss | folds={inner_cv_splits}"
    )

    for gi, params in enumerate(base_grid, start=1):
        fold_logloss = []
        fold_trees = []
        fold_nb_info = []

        for tr_i, va_i in folds:
            X_tr = X_train_enc[tr_i]
            y_tr = y_train[tr_i]

            X_va = X_train_enc[va_i]
            y_va = y_train[va_i]

            booster, best_trees, val_logloss = xgb_fit_baseline_with_es(
                X_tr,
                y_tr,
                X_va,
                y_va,
                params=params,
                seed=int(seed),
            )

            fold_logloss.append(float(val_logloss))
            fold_trees.append(int(best_trees))

            dvalid = xgb.DMatrix(X_va, label=y_va)
            margins = booster.predict(dvalid, output_margin=True)

            nb_val = net_benefit_hard_band_xgb(
                margins,
                y_va,
                t_min=float(t_min),
                t_max=float(t_max),
                n_grid=int(VAL_RANGE_POINTS),
                tau_eval=1.0,
                input_is_margin=True,
            )

            fold_nb_info.append(float(nb_val))

        mean_logloss = float(np.mean(fold_logloss))
        median_trees = int(np.median(fold_trees))
        mean_nb_info = float(np.mean(fold_nb_info))

        print(
            f"[Inner CV][Baseline {gi:03d}/{len(base_grid)}] "
            f"val_logloss={mean_logloss:.6f} | "
            f"val_nb(info)={mean_nb_info:+.6f} | "
            f"median_trees={median_trees:4d} | "
            f"max_depth={params['max_depth']} | "
            f"lr={params['learning_rate']:.2g} | "
            f"min_child_weight={params['min_child_weight']} | "
            f"reg_lambda={params['reg_lambda']} | "
            f"gamma={params['gamma']} | "
            f"subsample={params['subsample']} | "
            f"colsample={params['colsample_bytree']}"
        )

        if (best_base is None) or (mean_logloss < best_base[0]):
            best_base = (
                float(mean_logloss),
                dict(params),
                int(median_trees),
                float(mean_nb_info),
            )
            best_base_fold_trees = list(fold_trees)

    best_baseline_params = best_base[1]
    median_baseline_trees = int(np.median(best_base_fold_trees))

    print(
        f"\n[Inner CV][Baseline BEST] "
        f"val_logloss={best_base[0]:.6f} | "
        f"val_nb(info)={best_base[3]:+.6f} | "
        f"median_trees={median_baseline_trees} | "
        f"params={best_baseline_params}"
    )

    print(
        f"[Inner CV][Baseline BEST] "
        f"fold_specific_best_trees={best_base_fold_trees}"
    )

    # =========================================================
    # 2) Annealed NB selection by validation hard NB
    # =========================================================

    nb_grid = nb_hyperparameter_grid_annealed()
    best_nb = None

    print(
        f"\n[Inner CV][Annealed NB | h={hessian_mode}] "
        f"{len(nb_grid)} configs | metric=VAL hard NB | folds={inner_cv_splits}"
    )

    for gi, cfg in enumerate(nb_grid, start=1):
        fold_val_nb = []
        fold_added_total = []
        fold_added_by_inv_temp = {
            1.0: [],
            4.0: [],
            10.0: [],
        }

        for fj, (tr_i, va_i) in enumerate(folds):
            X_tr = X_train_enc[tr_i]
            y_tr = y_train[tr_i]

            X_va = X_train_enc[va_i]
            y_va = y_train[va_i]

            fold_baseline_trees = int(best_base_fold_trees[fj])

            warm0 = warm_start_baseline_train_only(
                X_tr,
                y_tr,
                baseline_params=best_baseline_params,
                baseline_trees=fold_baseline_trees,
                seed=int(seed),
            )

            params_nb_stage = _make_nb_stage_params_from_baseline(
                best_baseline_params,
                cfg,
                seed=int(seed),
            )

            _, added_best, val_nb, stage_info = nb_continue_annealed_with_stage_commit(
                booster0=warm0,
                X_train_enc=X_tr,
                y_train=y_tr,
                X_valid_enc=X_va,
                y_valid=y_va,
                t_min=float(t_min),
                t_max=float(t_max),
                anneal_inv_temps=tuple(cfg["anneal_inv_temps"]),
                num_points_train=int(TRAIN_RANGE_POINTS),
                n_grid_eval=int(VAL_RANGE_POINTS),
                additional_trees_cap=int(cfg["additional_trees_cap"]),
                eval_every_trees=int(cfg["eval_every_trees"]),
                params_nb_stage=params_nb_stage,
                hessian_mode=str(hessian_mode),
                cfg=cfg,
            )

            fold_val_nb.append(float(val_nb))
            fold_added_total.append(int(added_best))

            committed_by_temp = {
                float(info["inv_temp"]): int(info["committed_added"])
                for info in stage_info
            }

            for inv_temp in fold_added_by_inv_temp:
                fold_added_by_inv_temp[inv_temp].append(
                    int(committed_by_temp.get(float(inv_temp), 0))
                )

        mean_val_nb = float(np.mean(fold_val_nb))
        median_added_total = int(np.median(fold_added_total))

        median_added_it1 = int(np.median(fold_added_by_inv_temp[1.0]))
        median_added_it4 = int(np.median(fold_added_by_inv_temp[4.0]))
        median_added_it10 = int(np.median(fold_added_by_inv_temp[10.0]))

        print(
            f"[Inner CV][Annealed NB {gi:03d}/{len(nb_grid)} | h={hessian_mode}] "
            f"val_nb={mean_val_nb:+.6f} | "
            f"median_added_total={median_added_total:4d} | "
            f"it1={median_added_it1:4d} | "
            f"it4={median_added_it4:4d} | "
            f"it10={median_added_it10:4d} | "
            f"schedule={cfg['anneal_inv_temps']} | "
            f"max_depth={cfg['max_depth']} | "
            f"lr={cfg['learning_rate']:.2g} | "
            f"min_child_weight={cfg['min_child_weight']} | "
            f"reg_lambda={cfg['reg_lambda']} | "
            f"gamma={cfg['gamma']} | "
            f"subsample={cfg['subsample']} | "
            f"colsample={cfg['colsample_bytree']}"
        )

        if (best_nb is None) or (mean_val_nb > best_nb[0]):
            cfg_best = dict(cfg)
            cfg_best["_baseline_params_for_nb_stage"] = dict(best_baseline_params)
            cfg_best["median_added_trees"] = int(median_added_total)
            cfg_best["median_added_trees_it1"] = int(median_added_it1)
            cfg_best["median_added_trees_it4"] = int(median_added_it4)
            cfg_best["median_added_trees_it10"] = int(median_added_it10)

            best_nb = (
                float(mean_val_nb),
                cfg_best,
                int(median_added_total),
            )

    best_nb_cfg = best_nb[1]
    median_added_trees = int(best_nb[2])

    print(
        f"\n[Inner CV][Annealed NB BEST | h={hessian_mode}] "
        f"val_nb={best_nb[0]:+.6f} | "
        f"median_added={median_added_trees} | "
        f"cfg={best_nb_cfg}"
    )

    return (
        best_baseline_params,
        int(median_baseline_trees),
        best_nb_cfg,
        int(median_added_trees),
    )


def refit_nb_on_train_from_baseline_encoded(
    *,
    booster_baseline_train,
    X_train_enc,
    y_train,
    t_min: float,
    t_max: float,
    nb_cfg: dict,
    hessian_mode: str,
    seed: int,
):
    dtrain = xgb.DMatrix(
        X_train_enc,
        label=np.asarray(y_train, dtype=np.float32),
    )

    params_nb = _make_nb_stage_params_from_baseline(
        baseline_params=nb_cfg.get("_baseline_params_for_nb_stage"),
        nb_cfg=nb_cfg,
        seed=int(seed),
    )

    stage_tree_counts = {
        1.0: int(nb_cfg.get("median_added_trees_it1", 0)),
        4.0: int(nb_cfg.get("median_added_trees_it4", 0)),
        10.0: int(nb_cfg.get("median_added_trees_it10", 0)),
    }

    booster = booster_baseline_train

    for inv_temp in nb_cfg.get("anneal_inv_temps", NB_ANNEAL_INV_TEMPS):
        inv_temp = float(inv_temp)
        n_stage_trees = int(stage_tree_counts.get(inv_temp, 0))

        if n_stage_trees <= 0:
            continue

        objective = make_xgb_smooth_net_benefit_range_objective(
            t_min=float(t_min),
            t_max=float(t_max),
            num_points=int(TRAIN_RANGE_POINTS),
            inv_temp=float(inv_temp),
            hessian_mode=str(hessian_mode),
        )

        print(
            f"    [final refit] inv_temp={inv_temp:g} | "
            f"adding {n_stage_trees} trees"
        )

        booster = xgb.train(
            params=params_nb,
            dtrain=dtrain,
            num_boost_round=int(n_stage_trees),
            obj=objective,
            xgb_model=booster,
            verbose_eval=False,
        )

    return booster




## Step 10 — Main XGBoost TabZilla benchmark

This step executes the complete XGBoost benchmark across the predefined OpenML datasets.

For each eligible dataset, five outer train/test rotations are evaluated. Preprocessing is fitted using the outer training fold only.

Within each outer training fold, BCE-XGBoost hyperparameters are selected using stratified inner cross-validation and validation logloss. The selected BCE model is then refitted and used as the baseline model.

For each reference decision threshold, the following approaches are evaluated on the same independent outer test fold:

- BCE-trained XGBoost;
- BCE-trained XGBoost with local temperature scaling;
- BCE-trained XGBoost with local Platt scaling;
- Smooth Net Benefit-trained XGBoost using the `absw` Hessian;
- Smooth Net Benefit-trained XGBoost using the `fixed025` Hessian;
- Smooth Net Benefit-trained XGBoost using the `true` Hessian.

Smooth Net Benefit hyperparameters are selected within the outer training data using validation Net Benefit over the relevant threshold band. The BCE model serves as the warm start for all Smooth Net Benefit continuation models.

For every dataset, fold, threshold type, and model variant, the benchmark stores dataset characteristics, selected hyperparameters, number of trees, calibration diagnostics, test-set Net Benefit, and the difference in Net Benefit relative to the BCE baseline.

In [45]:
# ======================================================================
# Final XGBoost TabZilla runner with BCE calibration rows
#   - runs selected OpenML datasets
#   - runs 5 outer folds
#   - runs prevalence/inverse-prevalence threshold bands
#   - stores BCE-XGB, calibrated BCE-XGB, and annealed NB-XGB rows
#   - no CSV appending for now
# ======================================================================

import numpy as np
import pandas as pd
import xgboost as xgb


xgb_results = []


def add_xgb_tabzilla_result_row(
    *,
    meta: dict,
    run_id: int,
    threshold_name: str,
    model_name: str,
    y_train: np.ndarray,
    y_test: np.ndarray,
    t_ref: float,
    t_min: float,
    t_max: float,
    test_nb: float,
    delta_vs_bce: float,
    hessian_mode: str = "__BASELINE__",
    baseline_trees: int | None = None,
    baseline_params: dict | None = None,
    nb_added_trees: int | None = None,
    nb_added_trees_it1: int | None = None,
    nb_added_trees_it4: int | None = None,
    nb_added_trees_it10: int | None = None,
    nb_anneal_schedule=None,
    nb_params: dict | None = None,
    calibration_info: dict | None = None,
    n_features_encoded: int | None = None,
):
    calibration_info = calibration_info or {}

    xgb_results.append(
        {
            "openml_id": int(meta["openml_id"]),
            "name": str(meta["name"]),
            "run_id": int(run_id),

            "threshold_name": str(threshold_name),
            "run_type": str(threshold_name),

            "model_type": str(model_name),
            "model": str(model_name),
            "hessian_mode": str(hessian_mode),

            "n_rows": int(meta["n_rows"]),
            "n_features_raw": int(meta["n_features"]),
            "n_features_encoded": (
                int(n_features_encoded)
                if n_features_encoded is not None
                else np.nan
            ),

            "prevalence": float(meta["prevalence"]),
            "prev_train": float(np.mean(y_train)),
            "prev_test": float(np.mean(y_test)),

            "pos_label": str(meta.get("pos_label", "1")),
            "neg_label": str(meta.get("neg_label", "0")),

            "t_ref": float(t_ref),
            "threshold": float(t_ref),
            "t_min": float(t_min),
            "t_max": float(t_max),
            "band_min": float(t_min),
            "band_max": float(t_max),

            "test_nb": float(test_nb),
            "baseline_test_nb": (
                float(test_nb)
                if model_name == "bce_xgb"
                else np.nan
            ),
            "nb_test_nb": (
                float(test_nb)
                if model_name == "nb_xgb"
                else np.nan
            ),
            "delta_vs_bce": float(delta_vs_bce),
            "delta_nb_vs_baseline": (
                float(delta_vs_bce)
                if model_name == "nb_xgb"
                else np.nan
            ),

            "baseline_trees": (
                int(baseline_trees)
                if baseline_trees is not None
                else np.nan
            ),
            "baseline_params": (
                str(baseline_params)
                if baseline_params is not None
                else None
            ),

            "nb_added_trees": (
                int(nb_added_trees)
                if nb_added_trees is not None
                else np.nan
            ),
            "nb_added_trees_it1": (
                int(nb_added_trees_it1)
                if nb_added_trees_it1 is not None
                else np.nan
            ),
            "nb_added_trees_it4": (
                int(nb_added_trees_it4)
                if nb_added_trees_it4 is not None
                else np.nan
            ),
            "nb_added_trees_it10": (
                int(nb_added_trees_it10)
                if nb_added_trees_it10 is not None
                else np.nan
            ),
            "nb_anneal_schedule": (
                str(nb_anneal_schedule)
                if nb_anneal_schedule is not None
                else None
            ),
            "nb_params": (
                str(nb_params)
                if nb_params is not None
                else None
            ),

            "local_calib_low": calibration_info.get("local_calib_low", np.nan),
            "local_calib_high": calibration_info.get("local_calib_high", np.nan),
            "local_calib_half_width": calibration_info.get("local_calib_half_width", np.nan),
            "local_calib_n": calibration_info.get("local_calib_n", np.nan),
            "local_calib_n_pos": calibration_info.get("local_calib_n_pos", np.nan),
            "local_calib_n_neg": calibration_info.get("local_calib_n_neg", np.nan),
            "local_calib_met_minimum": calibration_info.get("local_calib_met_minimum", np.nan),
            "local_calib_used_full_range": calibration_info.get("local_calib_used_full_range", np.nan),

            "temperature": calibration_info.get("temp_temperature", np.nan),
            "temp_nll_before": calibration_info.get("temp_nll_before", np.nan),
            "temp_nll_after": calibration_info.get("temp_nll_after", np.nan),
            "temp_test_mask_n": calibration_info.get("temp_test_mask_n", np.nan),

            "platt_slope": calibration_info.get("platt_slope", np.nan),
            "platt_intercept": calibration_info.get("platt_intercept", np.nan),
            "platt_nll_before": calibration_info.get("platt_nll_before", np.nan),
            "platt_nll_after": calibration_info.get("platt_nll_after", np.nan),
            "platt_test_mask_n": calibration_info.get("platt_test_mask_n", np.nan),
        }
    )


def run_one_outer_fold_xgb_tabzilla(
    *,
    X_train_enc: np.ndarray,
    y_train: np.ndarray,
    X_test_enc: np.ndarray,
    y_test: np.ndarray,
    meta: dict,
    run_id: int,
    threshold_name: str,
    t_ref: float,
    t_min: float,
    t_max: float,
    hessian_modes: list[str] | None = None,
    inner_cv_splits: int = 5,
    seed: int = MODEL_SEED,
):
    if hessian_modes is None:
        hessian_modes = list(HESSIAN_MODES)

    X_train_enc = np.asarray(X_train_enc, dtype=np.float32)
    X_test_enc = np.asarray(X_test_enc, dtype=np.float32)

    y_train = np.asarray(y_train, dtype=np.float32).reshape(-1)
    y_test = np.asarray(y_test, dtype=np.float32).reshape(-1)

    selected_by_hessian = {}

    # --------------------------------------------------
    # Select baseline + NB configuration for each Hessian mode
    # --------------------------------------------------
    for h in hessian_modes:
        (
            best_baseline_params,
            median_baseline_trees,
            best_nb_cfg,
            median_added_trees,
        ) = select_baseline_and_nb_inner_cv(
            X_train_enc,
            y_train,
            t_min=float(t_min),
            t_max=float(t_max),
            hessian_mode=str(h),
            inner_cv_splits=int(inner_cv_splits),
            seed=int(seed),
        )

        selected_by_hessian[str(h)] = {
            "best_baseline_params": dict(best_baseline_params),
            "median_baseline_trees": int(median_baseline_trees),
            "best_nb_cfg": dict(best_nb_cfg),
            "median_added_trees": int(median_added_trees),
        }

    # --------------------------------------------------
    # Refit one BCE baseline on full outer-training data
    # --------------------------------------------------
    first_h = str(hessian_modes[0])

    best_baseline_params = selected_by_hessian[first_h]["best_baseline_params"]
    median_baseline_trees = selected_by_hessian[first_h]["median_baseline_trees"]

    booster_baseline_train = refit_baseline_on_trainval(
        X_train_enc,
        y_train,
        params=best_baseline_params,
        n_trees=int(median_baseline_trees),
        seed=int(seed),
    )

    dtrain = xgb.DMatrix(X_train_enc, label=y_train)
    dtest = xgb.DMatrix(X_test_enc, label=y_test)

    train_logits_base = booster_baseline_train.predict(
        dtrain,
        output_margin=True,
    )

    test_logits_base = booster_baseline_train.predict(
        dtest,
        output_margin=True,
    )

    # --------------------------------------------------
    # BCE-XGB uncalibrated test NB
    # --------------------------------------------------
    nb_bce = evaluate_nb_from_xgb_logits_np(
        test_logits_base,
        y_test,
        thresh_min=float(t_min),
        thresh_max=float(t_max),
        num_points=int(TEST_RANGE_POINTS),
    )

    print(
        f"\n[TEST][OpenML {meta['openml_id']} | run {run_id} | {threshold_name}] "
        f"BCE-XGB NB={nb_bce:+.6f} | trees={median_baseline_trees}"
    )

    add_xgb_tabzilla_result_row(
        meta=meta,
        run_id=run_id,
        threshold_name=threshold_name,
        model_name="bce_xgb",
        y_train=y_train,
        y_test=y_test,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce,
        delta_vs_bce=0.0,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
        n_features_encoded=X_train_enc.shape[1],
    )

    # --------------------------------------------------
    # Local temperature scaling and local Platt scaling
    # Only for BCE-XGB
    # --------------------------------------------------
    calib = evaluate_bce_xgb_local_calibration(
        train_logits_base=train_logits_base,
        y_train=y_train,
        test_logits_base=test_logits_base,
        y_test=y_test,
        t_ref=float(t_ref),
        t_min=float(t_min),
        t_max=float(t_max),
    )

    nb_bce_temp = float(calib["bce_nb_temp_scaled"])
    nb_bce_platt = float(calib["bce_nb_platt_scaled"])

    add_xgb_tabzilla_result_row(
        meta=meta,
        run_id=run_id,
        threshold_name=threshold_name,
        model_name="bce_xgb_local_temperature",
        y_train=y_train,
        y_test=y_test,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce_temp,
        delta_vs_bce=nb_bce_temp - nb_bce,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
        calibration_info=calib,
        n_features_encoded=X_train_enc.shape[1],
    )

    add_xgb_tabzilla_result_row(
        meta=meta,
        run_id=run_id,
        threshold_name=threshold_name,
        model_name="bce_xgb_local_platt",
        y_train=y_train,
        y_test=y_test,
        t_ref=t_ref,
        t_min=t_min,
        t_max=t_max,
        test_nb=nb_bce_platt,
        delta_vs_bce=nb_bce_platt - nb_bce,
        hessian_mode="__BASELINE__",
        baseline_trees=median_baseline_trees,
        baseline_params=best_baseline_params,
        calibration_info=calib,
        n_features_encoded=X_train_enc.shape[1],
    )

    print(
        f"[CAL][OpenML {meta['openml_id']} | run {run_id} | {threshold_name}] "
        f"BCE-temp Δ={nb_bce_temp - nb_bce:+.6f} | "
        f"BCE-Platt Δ={nb_bce_platt - nb_bce:+.6f}"
    )

    # --------------------------------------------------
    # NB-XGB final refit and test evaluation
    # No calibration is applied to NB-XGB logits
    # --------------------------------------------------
    for h in hessian_modes:
        sel = selected_by_hessian[str(h)]

        best_cfg = dict(sel["best_nb_cfg"])
        best_cfg["_baseline_params_for_nb_stage"] = dict(
            sel["best_baseline_params"]
        )

        booster_nb = refit_nb_on_train_from_baseline_encoded(
            booster_baseline_train=booster_baseline_train,
            X_train_enc=X_train_enc,
            y_train=y_train,
            t_min=float(t_min),
            t_max=float(t_max),
            nb_cfg=best_cfg,
            hessian_mode=str(h),
            seed=int(seed),
        )

        test_logits_nb = booster_nb.predict(
            dtest,
            output_margin=True,
        )

        nb_snb = evaluate_nb_from_xgb_logits_np(
            test_logits_nb,
            y_test,
            thresh_min=float(t_min),
            thresh_max=float(t_max),
            num_points=int(TEST_RANGE_POINTS),
        )

        delta = float(nb_snb - nb_bce)

        print(
            f"[TEST][OpenML {meta['openml_id']} | run {run_id} | "
            f"{threshold_name} | h={h}] "
            f"NB-XGB NB={nb_snb:+.6f} | Δ={delta:+.6f} | "
            f"base_trees={median_baseline_trees} + "
            f"added=({best_cfg.get('median_added_trees_it1', 0)}, "
            f"{best_cfg.get('median_added_trees_it4', 0)}, "
            f"{best_cfg.get('median_added_trees_it10', 0)})"
        )

        add_xgb_tabzilla_result_row(
            meta=meta,
            run_id=run_id,
            threshold_name=threshold_name,
            model_name="nb_xgb",
            y_train=y_train,
            y_test=y_test,
            t_ref=t_ref,
            t_min=t_min,
            t_max=t_max,
            test_nb=nb_snb,
            delta_vs_bce=delta,
            hessian_mode=str(h),
            baseline_trees=median_baseline_trees,
            baseline_params=sel["best_baseline_params"],
            nb_added_trees=int(best_cfg.get("median_added_trees", 0)),
            nb_added_trees_it1=int(best_cfg.get("median_added_trees_it1", 0)),
            nb_added_trees_it4=int(best_cfg.get("median_added_trees_it4", 0)),
            nb_added_trees_it10=int(best_cfg.get("median_added_trees_it10", 0)),
            nb_anneal_schedule=best_cfg.get("anneal_inv_temps", NB_ANNEAL_INV_TEMPS),
            nb_params={
                k: v
                for k, v in best_cfg.items()
                if not k.startswith("_")
            },
            n_features_encoded=X_train_enc.shape[1],
        )


def run_one_openml_dataset_xgb_tabzilla(
    *,
    did: int,
    outer_seed: int = SEED_GLOBAL,
    model_seed: int = MODEL_SEED,
    hessian_modes: list[str] | None = None,
    inner_cv_splits: int = 5,
):
    ds, X_df, y, meta = load_openml_dataset(int(did))

    if int(meta["n_rows"]) < int(MIN_ROWS):
        print(
            f"[SKIP] OpenML {did} | {meta['name']} | "
            f"n_rows={meta['n_rows']} < MIN_ROWS={MIN_ROWS}"
        )
        return None

    folds = make_5fold_indices(
        y,
        seed=int(outer_seed),
    )

    print(
        f"\n==================== OpenML {meta['openml_id']} | "
        f"{meta['name']} ===================="
    )

    print(
        f"n_rows={meta['n_rows']} | "
        f"n_features={meta['n_features']} | "
        f"prevalence={meta['prevalence']:.4f} | "
        f"pos_label={meta['pos_label']} | neg_label={meta['neg_label']}"
    )

    for run_id in range(5):
        train_inds, test_inds = indices_for_run_train_test(
            folds,
            run_id=int(run_id),
        )

        X_tr = X_df.iloc[train_inds].copy()
        X_te = X_df.iloc[test_inds].copy()

        y_tr = np.asarray(y[train_inds], dtype=np.float32).reshape(-1)
        y_te = np.asarray(y[test_inds], dtype=np.float32).reshape(-1)

        bundle, X_tr_enc, X_te_enc = fit_train_preprocessor_and_transform(
            X_tr,
            X_te,
        )

        prev_train = float(np.mean(y_tr))

        threshold_specs = threshold_specs_from_prevalence(
            prev_train,
            mid_prev_low=MID_PREV_LOW,
            mid_prev_high=MID_PREV_HIGH,
        )

        print(
            f"\n[OpenML {meta['openml_id']} | run {run_id}] "
            f"train_n={len(y_tr)} | test_n={len(y_te)} | "
            f"prev_train={prev_train:.4f} | "
            f"encoded_features={X_tr_enc.shape[1]}"
        )

        for spec in threshold_specs:
            threshold_name = str(spec["threshold_name"])
            t_ref = float(spec["threshold"])
            t_min, t_max = band_from_t_ref(
                t_ref,
                half_width=BAND_HALF_WIDTH,
            )

            print(
                f"\n[OpenML {meta['openml_id']} | run {run_id}] "
                f"=== THRESHOLD: {threshold_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            run_one_outer_fold_xgb_tabzilla(
                X_train_enc=X_tr_enc,
                y_train=y_tr,
                X_test_enc=X_te_enc,
                y_test=y_te,
                meta=meta,
                run_id=int(run_id),
                threshold_name=threshold_name,
                t_ref=float(t_ref),
                t_min=float(t_min),
                t_max=float(t_max),
                hessian_modes=hessian_modes,
                inner_cv_splits=int(inner_cv_splits),
                seed=int(model_seed) + int(run_id),
            )

    return pd.DataFrame(xgb_results)


def run_openml_datasets_xgb_tabzilla(
    openml_ids: list[int],
    *,
    outer_seed: int = SEED_GLOBAL,
    model_seed: int = MODEL_SEED,
    hessian_modes: list[str] | None = None,
    inner_cv_splits: int = 5,
):
    global xgb_results
    xgb_results = []

    for did in openml_ids:
        run_one_openml_dataset_xgb_tabzilla(
            did=int(did),
            outer_seed=int(outer_seed),
            model_seed=int(model_seed),
            hessian_modes=hessian_modes,
            inner_cv_splits=int(inner_cv_splits),
        )

    df_results_xgb = pd.DataFrame(xgb_results)

    print("\nFinished.")
    print("df_results_xgb shape:", df_results_xgb.shape)

    return df_results_xgb

## Step 11 — Execute the full benchmark

This cell starts the full TabZilla XGBoost benchmark using the predefined OpenML dataset list.

All three Smooth Net Benefit Hessian variants are evaluated, and model selection uses five inner cross-validation folds within each outer training fold.

Because this stage performs nested XGBoost hyperparameter selection and decision-focused continuation for multiple datasets, folds, thresholds, and Hessian variants, it is the most computationally intensive part of the notebook.

In [46]:
df_results_xgb = run_openml_datasets_xgb_tabzilla(
    openml_ids=OPENML_IDS,
    hessian_modes=HESSIAN_MODES,
    inner_cv_splits=5,
)


==================== OpenML 1120 | MagicTelescope ====================
n_rows=19020 | n_features=10 | prevalence=0.3516 | pos_label=h | neg_label=g

[OpenML 1120 | run 0] train_n=15216 | test_n=3804 | prev_train=0.3517 | encoded_features=10

[OpenML 1120 | run 0] === THRESHOLD: prev (t_ref=0.3517, [0.3267, 0.3767]) ===

[Inner CV][Baseline] 32 configs | metric=VAL logloss | folds=5
[Inner CV][Baseline 001/32] val_logloss=0.294017 | val_nb(info)=+0.257259 | median_trees= 997 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[Inner CV][Baseline 002/32] val_logloss=0.305039 | val_nb(info)=+0.252557 | median_trees= 465 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=1.0 | subsample=1.0 | colsample=1.0
[Inner CV][Baseline 003/32] val_logloss=0.297111 | val_nb(info)=+0.255838 | median_trees= 990 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=5.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[Inner C

KeyboardInterrupt: 